# Hands-On 9: Build Advanced Research Workflow

We build a workflow with subgraphs, parallel tasks, validation, retry routing, and error handling.

#### Imports

We import `TypedDict` for state schemas, `Literal` for routing return types, and `StateGraph`, `START`, `END` from LangGraph.

In [1]:
from typing import TypedDict, Literal

import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END

In [2]:
load_dotenv(r"D:\Agent_Flow\.env", override=True)

print(
    "OpenRouter API key loaded:",
    bool(os.getenv("OPENROUTER_API_KEY"))
)

OpenRouter API key loaded: True


In [3]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

print("OpenRouter LLM configured successfully")

OpenRouter LLM configured successfully


In [4]:
response = llm.invoke(
    "Give me one sentence about quantum computing."
)

print(response.content)

Quantum computing harnesses the principles of quantum mechanics to process information in fundamentally different ways than classical computers, enabling the potential for exponentially faster problem-solving capabilities for certain complex tasks.


#### Define Main State

We define `ResearchState` with fields for topic, sources, tasks, results, validation flag, and final summary.

In [5]:
class ResearchState(TypedDict):
    topic: str
    sources: list
    tasks: list
    results: list
    validated: bool
    summary: str
    retry_count: int

#### Create Research Subgraph

A subgraph is a smaller LangGraph that can be invoked as one unit. Here it builds a query, fetches data, and parses results.

In [6]:
class SubgraphState(TypedDict):
    topic: str
    query: str
    raw_data: str
    parsed: str

### Query Builder

In [7]:
def query_builder(state: SubgraphState):
    prompt = f"""
Create a concise research query for the topic:

{state['topic']}

The query should help find useful information for a research workflow.
Return only the query.
"""

    response = llm.invoke(prompt)

    return {
        "query": response.content.strip()
    }

#### Fetch Research Data

Because we aren't connecting an external search API in this exercise, we'll use the LLM as the research-data generator

In [8]:
def fetch_data(state: SubgraphState):
    prompt = f"""
You are a research assistant.

Research the following topic using this query:

Topic:
{state['topic']}

Query:
{state['query']}

Provide factual, structured research information including:
- Key concepts
- Important developments
- Applications
- Benefits
- Challenges

Do not invent specific citations or sources.
"""

    response = llm.invoke(prompt)

    return {
        "raw_data": response.content
    }

### Parse Results

In [10]:
def parse_results(state: SubgraphState):
    prompt = f"""
Extract the most important research findings from the following material.

Topic:
{state['topic']}

Research:
{state['raw_data']}

Return 5 concise key findings.
"""

    response = llm.invoke(prompt)

    return {
        "parsed": response.content
    }

#### Build Research Subgraph

In [11]:
sub_graph = StateGraph(SubgraphState)

sub_graph.add_node(
    "query_builder",
    query_builder
)

sub_graph.add_node(
    "fetch_data",
    fetch_data
)

sub_graph.add_node(
    "parse_results",
    parse_results
)

sub_graph.add_edge(
    START,
    "query_builder"
)

sub_graph.add_edge(
    "query_builder",
    "fetch_data"
)

sub_graph.add_edge(
    "fetch_data",
    "parse_results"
)

sub_graph.add_edge(
    "parse_results",
    END
)

research_subgraph = sub_graph.compile()

print("Research subgraph created successfully")

Research subgraph created successfully


#### Run Parallel Tasks

This node creates three independent task descriptions, then runs the research subgraph for each one to simulate parallel execution.

### Parallel Task Node

In [12]:
def run_parallel_tasks(state: ResearchState):
    
    tasks = [
        f"Literature review on {state['topic']}",
        f"Data and technical analysis of {state['topic']}",
        f"Real-world applications and case studies of {state['topic']}"
    ]

    print(
        f"→ Launching {len(tasks)} research tasks..."
    )

    results = []

    for task in tasks:
        
        print(f"\n🔍 Running: {task}")

        sub_input = {
            "topic": f"{state['topic']} - {task}",
            "query": "",
            "raw_data": "",
            "parsed": ""
        }

        sub_result = research_subgraph.invoke(
            sub_input
        )

        results.append(
            f"{task}:\n{sub_result['parsed']}"
        )

    return {
        "tasks": tasks,
        "results": results
    }

#### Aggregate Results

Combine all parallel outputs into a single draft summary for validation.

In [13]:
def aggregate_node(state: ResearchState):
    
    combined = "\n\n".join(
        state["results"]
    )

    prompt = f"""
You are an aggregation agent.

Topic:
{state['topic']}

The following are independent research results:

{combined}

Combine them into one coherent research draft.

Remove duplicate information.
Highlight important findings.
Do not add unsupported facts.
"""

    response = llm.invoke(prompt)

    print("→ Aggregating research results...")

    return {
        "results": state["results"],
        "summary": response.content
    }

#### Add Validation Node

Validate whether the aggregated results pass quality checks (summary non-empty and at least 2 results).

In [14]:
def validate_node(state: ResearchState):
    
    summary = state.get("summary", "")
    results = state.get("results", [])

    basic_valid = (
        bool(summary.strip())
        and len(results) >= 2
    )

    if not basic_valid:
        print("→ Validation: ❌ failed")
        return {
            "validated": False
        }

    prompt = f"""
Validate this research summary.

Topic:
{state['topic']}

Summary:
{summary}

Check whether:
1. The summary is relevant to the topic.
2. It contains useful research findings.
3. It is coherent.
4. It does not appear empty or incomplete.

Return only:
PASS
or
FAIL
"""

    response = llm.invoke(prompt)

    valid = "PASS" in response.content.upper()

    print(
        f"→ Validation: "
        f"{'✅ passed' if valid else '❌ failed'}"
    )

    return {
        "validated": valid
    }

#### Add Retry Routing

If validation fails, the router sends execution back to the parallel tasks node for a retry.

In [15]:
def route_after_validation(
    state: ResearchState
) -> Literal[
    "run_parallel_tasks",
    "finalize"
]:
    
    if state.get("validated"):
        return "finalize"

    if state.get("retry_count", 0) >= 2:
        print(
            "⚠️ Maximum validation retries reached."
        )
        return "finalize"

    return "run_parallel_tasks"

### Increment Retry Count

Because we added retry_count, update the research task node slightly.

In [16]:
def run_parallel_tasks(state: ResearchState):
    
    retry_count = state.get("retry_count", 0)

    if retry_count > 0:
        print(
            f"🔄 Research retry #{retry_count}"
        )

    tasks = [
        f"Literature review on {state['topic']}",
        f"Data and technical analysis of {state['topic']}",
        f"Real-world applications and case studies of {state['topic']}"
    ]

    print(
        f"→ Launching {len(tasks)} research tasks..."
    )

    results = []

    for task in tasks:

        print(f"\n🔍 Running: {task}")

        sub_input = {
            "topic": f"{state['topic']} - {task}",
            "query": "",
            "raw_data": "",
            "parsed": ""
        }

        sub_result = research_subgraph.invoke(
            sub_input
        )

        results.append(
            f"{task}:\n{sub_result['parsed']}"
        )

    return {
        "tasks": tasks,
        "results": results,
        "retry_count": retry_count + 1
    }

#### Finalize Node

When validation passes, the finalize node produces the final research summary.

In [17]:
def finalize_node(state: ResearchState):
    
    if not state.get("validated"):
        return {
            "summary": (
                f"⚠️ FINAL RESEARCH SUMMARY on "
                f"'{state['topic']}':\n\n"
                f"{state['summary']}\n\n"
                f"Note: Maximum validation retries "
                f"were reached."
            )
        }

    return {
        "summary": (
            f"✅ FINAL RESEARCH SUMMARY on "
            f"'{state['topic']}':\n\n"
            f"{state['summary']}"
        )
    }

#### Build Main Graph

We add all nodes and wire them with edges. The validation node uses a conditional edge to route back for retry or forward to finalize.

In [18]:
graph = StateGraph(ResearchState)

graph.add_node(
    "run_parallel_tasks",
    run_parallel_tasks
)

graph.add_node(
    "aggregate",
    aggregate_node
)

graph.add_node(
    "validate",
    validate_node
)

graph.add_node(
    "finalize",
    finalize_node
)

graph.add_edge(
    START,
    "run_parallel_tasks"
)

graph.add_edge(
    "run_parallel_tasks",
    "aggregate"
)

graph.add_edge(
    "aggregate",
    "validate"
)

graph.add_conditional_edges(
    "validate",
    route_after_validation,
    {
        "run_parallel_tasks": "run_parallel_tasks",
        "finalize": "finalize"
    }
)

graph.add_edge(
    "finalize",
    END
)

print("Advanced research graph created successfully")

Advanced research graph created successfully


In [19]:
workflow = graph.compile()

print("Advanced research workflow compiled successfully")

Advanced research workflow compiled successfully


#### Handle Failures

We wrap invocation in a retry loop that catches exceptions and returns a fallback message if all attempts fail.

In [20]:
def invoke_with_retry(
    graph,
    state,
    max_retries=3
):
    
    for attempt in range(max_retries):
        
        try:
            print(
                f"\n=== Workflow Attempt "
                f"{attempt + 1} ==="
            )

            return graph.invoke(state)

        except Exception as e:
            
            print(
                f"⚠️ Attempt {attempt + 1} failed: {e}"
            )

            if attempt == max_retries - 1:
                
                return {
                    **state,
                    "summary": (
                        "⚠️ Workflow failed after "
                        "all retry attempts."
                    )
                }

    return {
        **state,
        "summary": "⚠️ Workflow failed."
    }

#### Execute Workflow

Invoke the full workflow with retry protection using an initial state.

In [21]:
initial_state: ResearchState = {
    "topic": "Quantum Computing",
    "sources": [],
    "tasks": [],
    "results": [],
    "validated": False,
    "summary": "",
    "retry_count": 0
}

### Run Workflow

In [22]:
result = invoke_with_retry(
    workflow,
    initial_state
)

print("\nWorkflow execution completed.")


=== Workflow Attempt 1 ===
→ Launching 3 research tasks...

🔍 Running: Literature review on Quantum Computing

🔍 Running: Data and technical analysis of Quantum Computing

🔍 Running: Real-world applications and case studies of Quantum Computing
→ Aggregating research results...
→ Validation: ✅ passed

Workflow execution completed.


#### Generate Final Summary

Inspect the final workflow result.

In [23]:
print("=== RESEARCH TASKS ===")

for task in result["tasks"]:
    print("-", task)

=== RESEARCH TASKS ===
- Literature review on Quantum Computing
- Data and technical analysis of Quantum Computing
- Real-world applications and case studies of Quantum Computing


In [24]:
print("=== RESEARCH RESULTS ===")

for i, research in enumerate(
    result["results"],
    start=1
):
    print(f"\n--- Result {i} ---")
    print(research)

=== RESEARCH RESULTS ===

--- Result 1 ---
Literature review on Quantum Computing:
1. **Quantum Supremacy**: Google achieved quantum supremacy in 2019, demonstrating that quantum computers can outperform classical supercomputers in specific tasks, marking a significant milestone in the field.

2. **Quantum Error Correction**: The development of quantum error correction techniques is crucial for protecting quantum information from errors, enabling the construction of reliable quantum computers.

3. **Applications in Cryptography and Drug Discovery**: Quantum computing offers revolutionary applications, such as quantum key distribution for secure communications and accelerated drug discovery through accurate molecular simulations.

4. **Exponential Speedup and Enhanced Simulation**: Quantum computers can provide exponential speedup for certain problems and simulate quantum systems more accurately than classical computers, benefiting fields like chemistry and materials science.

5. **Chal

In [25]:
print("=== VALIDATION ===")

print(
    "Validated:",
    result["validated"]
)

print(
    "Retry count:",
    result["retry_count"]
)

=== VALIDATION ===
Validated: True
Retry count: 1


In [26]:
print("=== FINAL OUTPUT ===")

print(f"Topic: {result['topic']}")
print()
print(result["summary"])

=== FINAL OUTPUT ===
Topic: Quantum Computing

✅ FINAL RESEARCH SUMMARY on 'Quantum Computing':

# Research Draft on Quantum Computing

## Introduction
Quantum computing represents a transformative shift in computational capabilities, leveraging the principles of quantum mechanics to solve complex problems more efficiently than classical computers. This draft synthesizes recent research findings across various aspects of quantum computing, including its achievements, applications, challenges, and future prospects.

## Key Achievements
1. **Quantum Supremacy**: In 2019, Google achieved a significant milestone by demonstrating quantum supremacy, showcasing that quantum computers can outperform classical supercomputers in specific tasks. This achievement marks a pivotal moment in the evolution of quantum technology.

2. **Quantum Error Correction**: The development of quantum error correction techniques is essential for protecting quantum information from errors, enabling the construction

### Test Another Topic
— Generative AI

In [27]:
initial_state_2: ResearchState = {
    "topic": "Generative AI in Enterprise Applications",
    "sources": [],
    "tasks": [],
    "results": [],
    "validated": False,
    "summary": "",
    "retry_count": 0
}

result2 = invoke_with_retry(
    workflow,
    initial_state_2
)

print(result2["summary"])


=== Workflow Attempt 1 ===
→ Launching 3 research tasks...

🔍 Running: Literature review on Generative AI in Enterprise Applications

🔍 Running: Data and technical analysis of Generative AI in Enterprise Applications

🔍 Running: Real-world applications and case studies of Generative AI in Enterprise Applications
→ Aggregating research results...
→ Validation: ✅ passed
✅ FINAL RESEARCH SUMMARY on 'Generative AI in Enterprise Applications':

# Research Draft: Generative AI in Enterprise Applications

## Introduction
Generative AI has emerged as a transformative technology in enterprise applications, driven by advancements in algorithms, computational power, and the development of large language models (LLMs). This research synthesizes findings from literature reviews, technical analyses, and real-world case studies to provide a comprehensive overview of the current landscape of generative AI in enterprises.

## Key Findings

### 1. Advancements in Algorithms and Computational Power
The 

In [28]:
print("=== HANDS-ON 9 COMPLETE ===")

print("✓ OpenRouter API configured")
print("✓ Research subgraph created")
print("✓ Multiple research tasks executed")
print("✓ Results aggregated")
print("✓ Validation node added")
print("✓ Retry routing added")
print("✓ Final summary generated")
print("✓ Workflow failure handling added")
print("✓ Multiple topics tested")

=== HANDS-ON 9 COMPLETE ===
✓ OpenRouter API configured
✓ Research subgraph created
✓ Multiple research tasks executed
✓ Results aggregated
✓ Validation node added
✓ Retry routing added
✓ Final summary generated
✓ Workflow failure handling added
✓ Multiple topics tested


#### Summary

Advanced workflows combine subgraphs for encapsulation, parallel tasks for performance, validation for quality, and retry routing for robustness.